## Changing data — `MERGE INTO`, `UPDATE`, `DELETE`

Delta supports row-level changes on immutable object storage. The workhorse is **`MERGE INTO`** — the most-used Delta DML in production. It joins a *source* of incoming changes to the *target* table on a key and says what to do in each case:

```sql
MERGE INTO silver.customers AS t
USING staging_customers      AS s
   ON t.customer_id = s.customer_id
WHEN MATCHED AND s.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE   -- optional hard-delete
```

Two patterns the bank leans on:

- **SCD Type 1 (overwrite)** — a customer's `city` changes; update in place with `WHEN MATCHED ... UPDATE SET *`. No history kept.
- **SCD Type 2 (history)** — close the old row (`is_current = false`, `valid_to = today`) and insert a new active row. `APPLY CHANGES INTO` (module 05) is a declarative wrapper over this same machinery.

Plain **`UPDATE`** and **`DELETE`** work too, with two strategies underneath:

- **Copy-on-write** — Delta rewrites each Parquet file that holds matching rows. Correct, but expensive when only a few rows in a big file change.
- **Deletion vectors** (modern default) — Delta writes a small bitmap saying *"skip rows 17 and 234 in this file."* The Parquet file stays put; reads apply the bitmap on the fly, and the next `OPTIMIZE` resolves it for real.

For the bank, deletion vectors turn a GDPR "right to be forgotten" delete from a multi-gigabyte rewrite into a tiny commit.